# YOLOv26n Detection + Per-Lamp Tracking Training

This notebook trains the small YOLOv26 detector on red/white lamp detections and uses the sequence tracking annotations to detect red/white transitions over time. Transitions are not YOLO classes.

## How to run

- Environment: project `.venv` with editable install of `papi-detection`, plus a working Torch/Ultralytics CUDA stack.
- Working dir: any (the setup cell walks up to find `pyproject.toml`).
- Required local data: `data/datasets/papi_lamp_sequences/` (canonical day/night sequence dataset) and `models/base/yolo26n.pt`.
- Produces: rebuilt tracking annotations, the `yolo26n_combined` training config, and (when `RUN_TRAIN=True`) a new run under `runs/papi/yolo26n_sequence_red_white_safe/`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def _find_repo_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("repo root with pyproject.toml not found")


REPO = _find_repo_root(Path.cwd())
PYTHON = REPO / ".venv" / "Scripts" / "python.exe"
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

SEQUENCE_DATASET = REPO / "data" / "datasets" / "papi_lamp_sequences"
TRAINING_DATASET = SEQUENCE_DATASET / "yolo26n_combined"
BASE_MODEL = REPO / "models" / "base" / "yolo26n.pt"
RUN_NAME = "yolo26n_sequence_red_white_safe"
RUN_DIR = REPO / "runs" / "papi" / RUN_NAME
BEST_MODEL = RUN_DIR / "weights" / "best.pt"


def run(*args):
    command = [str(PYTHON), *args]
    print("$", " ".join(command))
    return subprocess.run(command, cwd=REPO, check=True)

## 1. Build And Validate Dataset Artifacts

Rebuild tracking annotations and the no-copy YOLO train/val/test config from the canonical day/night sequence folders.

In [ ]:
run("workflows/scripts/build_sequence_tracking.py")
run("workflows/scripts/prepare_yolo_sequence_dataset.py")

tracking_manifest = json.loads((SEQUENCE_DATASET / "tracking_manifest.json").read_text(encoding="utf-8"))
training_manifest = json.loads((TRAINING_DATASET / "manifest.json").read_text(encoding="utf-8"))
print(json.dumps({"tracking": tracking_manifest["totals"], "splits": training_manifest["splits"]}, indent=2))
assert tracking_manifest["error_count"] == 0


## 2. YOLOv26n Environment Smoke Check

The default base is `models/base/yolo26n.pt`, the small local YOLOv26 weight. Install or restore it before training if this cell cannot load the model.

In [ ]:
model = None
try:
    import torch
    from ultralytics import YOLO

    print("cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device", torch.cuda.get_device_name(0))
    model = YOLO(str(BASE_MODEL) if BASE_MODEL.exists() else "yolo26n.pt")
    print("loaded", BASE_MODEL if BASE_MODEL.exists() else "yolo26n.pt")
except Exception as exc:
    print("YOLOv26n smoke check failed:", repr(exc))
    print("Fix Torch/Ultralytics before setting RUN_TRAIN=True.")


## 3. Train Small Detector

The detector trains red/white boxes only; transition detection uses tracking annotations after inference.

**Heavy operation — set `RUN_TRAIN=True` (cell below) only when ready to fine-tune on GPU.**

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    assert model is not None, "Run and pass the YOLOv26n smoke check before training."
    model.train(
        data=str(TRAINING_DATASET / "data.yaml"),
        project=str(REPO / "runs" / "papi"),
        name=RUN_NAME,
        imgsz=1280,
        batch=2,
        epochs=100,
        patience=20,
        device=0,
        amp=False,
        exist_ok=True,
        seed=42,
        workers=0,
    )
else:
    print("Training disabled. Set RUN_TRAIN=True to fine-tune YOLOv26n.")


## 4. Validate Checkpoint And Inspect Transitions

In [ ]:
if BEST_MODEL.exists():
    trained = YOLO(str(BEST_MODEL))
    metrics = trained.val(data=str(TRAINING_DATASET / "data.yaml"), split="val", imgsz=1280, batch=2)
    print(metrics)
else:
    print("No trained checkpoint yet:", BEST_MODEL)

for regime in ["daytime", "nighttime"]:
    regime_root = SEQUENCE_DATASET / regime
    first_video = next(path for path in sorted(regime_root.iterdir()) if path.is_dir())
    transitions = (first_video / "transitions.csv").read_text(encoding="utf-8").splitlines()[:6]
    print(regime, first_video.name)
    print("\n".join(transitions) if transitions else "no transitions in first video")


## 5. Quality Gates

In [ ]:
run("-m", "json.tool", "workflows/notebooks/03_yolov26n_detection_tracking_training.ipynb")
run("-m", "pytest")
run("-m", "ruff", "check", "packages/papi/src", "packages/papi/tests", "workflows/scripts")